# 🧠 EIGENMODE ANALYSIS — Figure Generation Pipeline V3

**Purpose**: Generate publication-quality figures for eigenvalue/eigenmode analysis comparing Gifted and Control groups.

---

## Structure of NPZ Files

Each `.npz` file contains:
- `evals`: **(N_segments, N_channels)** complex eigenvalues for each time window
- `evecs`: **(N_segments, N_channels, N_channels)** eigenvectors
- `id`: Subject identifier (string)
- `group`: 'aacc' or 'control'
- `cond`: 'basal' or 'pvt'
- `win_sec`: Window size in seconds
- `sfreq`: Sampling frequency (256 Hz)
- `ch_names`: Channel names

**Key insight**: Each file has ~4000 time segments, each with 32 eigenvalues.

**Color convention**: 🔴 Gifted · 🔵 Control · Solid=PVT · Dashed=BASAL

---

This notebook is part of the public analysis repository associated with the EEG eigenmode study. It uses precomputed feature tables generated by the preprocessing and feature extraction scripts. Raw EEG recordings and participant-level data are not included because the study involves minors and is subject to ethical and privacy restrictions.

---
# SECTION 0 — Setup & Configuration
> ⚠️ **Run this section first**

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# IMPORTS
# ══════════════════════════════════════════════════════════════════════════════

from pathlib import Path
import warnings
import time
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Circle
from matplotlib import cm
from scipy.stats import gaussian_kde

warnings.filterwarnings("ignore")

# Set matplotlib parameters for publication quality
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.size"] = 10
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 9
plt.rcParams["legend.fontsize"] = 9

print("✅ Imports loaded successfully")

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# ── Colors (ALWAYS consistent) ────────────────────────────────────────────────
C_GIFTED = "#E05C3A"   # Orange-red for Gifted (AACC)
C_CTRL   = "#4878CF"   # Blue for Control
PALETTE  = {"Gifted": C_GIFTED, "Control": C_CTRL}

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE = Path(r"")

RUTAS_NPZ = {
    "all_channels":      BASE / "EIGENMODE_PIPELINE_PRE_ALL_CHANNELS"      / "03_eigs_npz",
    "no_occipital": BASE / "results/eigenmodes/no_occipital" / "03_eigs_npz",
}

RUTA_DMSD   = BASE / "results/features/dominant_mode_spatial_distribution" / "01_features"
RUTA_FIGS = BASE / "FIGURAS_V3"
RUTA_FIGS.mkdir(parents=True, exist_ok=True)

# ── Analysis Parameters ───────────────────────────────────────────────────────
FS = 256.0                    # Sampling frequency (Hz)
TAU_IMAG = 0.20              # Threshold for oscillatory mode (imaginary part)
PRIORITY_WINDOWS = [8.0, 4.0]  # Priority window sizes (seconds)

# ── Helper functions ──────────────────────────────────────────────────────────
def tsec(t0): 
    """Time elapsed since t0"""
    return f"{time.time()-t0:.1f}s"

def ensure_dir(path):
    """Create directory if it doesn't exist"""
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path

def save_fig(fig, filepath, dpi=300):
    """Save figure with consistent settings"""
    filepath = Path(filepath)
    ensure_dir(filepath.parent)
    fig.savefig(filepath, bbox_inches="tight", dpi=dpi, facecolor="white")
    print(f"   💾 {filepath.relative_to(RUTA_FIGS)}")

def to_scalar(val):
    """Convert numpy arrays to scalar values"""
    if isinstance(val, np.ndarray):
        if val.size == 0:
            return None
        return val.item() if val.ndim == 0 or val.size == 1 else val.flat[0]
    return val

def standardize_group(grupo):
    """Convert group to standard name"""
    grupo = str(to_scalar(grupo)).lower().strip()
    if grupo in ['aacc', '1', '1.0']:
        return 'Gifted'
    elif grupo in ['control', '0', '0.0', 'ctrl']:
        return 'Control'
    return grupo.capitalize()

print("✅ Configuration loaded")
print(f"   Output folder: {RUTA_FIGS}")
print(f"   Data sources: {list(RUTAS_NPZ.keys())}")

---
# SECTION 1 — Scan & Load NPZ Files
> Recursively find all .npz files and load metadata

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# SCAN ALL NPZ FILES RECURSIVELY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("SCANNING NPZ FILES")
print("="*70 + "\n")

inventory = []

for representation, folder in RUTAS_NPZ.items():
    if not folder.exists():
        print(f"⚠️  Folder not found: {folder}")
        continue
    
    # Find all .npz files recursively (including in win_X.Xs subfolders)
    npz_files = sorted(folder.rglob("*.npz"))
    
    print(f"{representation:15s}: {len(npz_files):4d} files found")
    
    for fpath in npz_files:
        try:
            data = np.load(fpath, allow_pickle=True)
            
            # Extract metadata
            grupo = standardize_group(data.get('group', data.get('grupo', 'unknown')))
            cond = str(to_scalar(data.get('cond', 'unknown'))).lower()
            win_sec = to_scalar(data.get('win_sec', None))
            subj_id = str(to_scalar(data.get('id', data.get('subj', 'unknown'))))
            
            # Get eigenvalue dimensions
            evals = data.get('evals', None)
            n_segments = evals.shape[0] if evals is not None else 0
            n_eigs_per_seg = evals.shape[1] if evals is not None and evals.ndim > 1 else 0
            
            inventory.append({
                'representation': representation,
                'filepath': fpath,
                'filename': fpath.name,
                'grupo': grupo,
                'cond': cond,
                'win_sec': win_sec,
                'subj_id': subj_id,
                'n_segments': n_segments,
                'n_eigs_per_seg': n_eigs_per_seg,
                'total_eigs': n_segments * n_eigs_per_seg if evals is not None else 0
            })
            
            data.close()
            
        except Exception as e:
            print(f"⚠️  Error reading {fpath.name}: {e}")

df_inventory = pd.DataFrame(inventory)

print(f"\n{'─'*70}")
print(f"✅ Total files found: {len(df_inventory)}")

if len(df_inventory) > 0:
    print(f"\nBy representation:")
    print(df_inventory.groupby('representation').size())
    print(f"\nBy group:")
    print(df_inventory.groupby('grupo').size())
    print(f"\nBy condition:")
    print(df_inventory.groupby('cond').size())
    print(f"\nBy window size:")
    print(df_inventory.groupby('win_sec').size())
    
    print(f"\nTotal eigenvalues across all files: {df_inventory['total_eigs'].sum():,}")
    
    # Save inventory
    inv_path = RUTA_FIGS / "_file_inventory.csv"
    df_inventory.to_csv(inv_path, index=False)
    print(f"\n💾 Inventory saved: {inv_path}")
else:
    print("\n⚠️  No files found!")

---
# SECTION 2 — Helper Functions
> Functions to process eigenvalues

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def load_npz_data(npz_path):
    """
    Load eigenvalues and metadata from NPZ file.
    
    Returns
    -------
    dict with:
        - evals_flat: all eigenvalues flattened to 1D array
        - evals_2d: (n_segments, n_channels) original shape
        - metadata: dict with grupo, cond, win_sec, subj_id
    """
    data = np.load(npz_path, allow_pickle=True)
    
    # Load eigenvalues (shape: n_segments x n_channels)
    evals_2d = data['evals']
    evals_flat = evals_2d.flatten()  # All eigenvalues in one array
    
    # Extract metadata
    metadata = {
        'grupo': standardize_group(data.get('group', data.get('grupo', 'unknown'))),
        'cond': str(to_scalar(data.get('cond', 'unknown'))).lower(),
        'win_sec': to_scalar(data.get('win_sec', None)),
        'subj_id': str(to_scalar(data.get('id', data.get('subj', 'unknown')))),
        'sfreq': to_scalar(data.get('sfreq', FS)),
        'n_segments': evals_2d.shape[0],
        'n_channels': evals_2d.shape[1]
    }
    
    data.close()
    
    return {
        'evals_flat': evals_flat,
        'evals_2d': evals_2d,
        'metadata': metadata
    }


def eigenvalue_features(evals, fs=256.0):
    """
    Compute features from complex eigenvalues.
    
    Parameters
    ----------
    evals : array of complex
        Eigenvalues (any shape, will be flattened)
    fs : float
        Sampling frequency in Hz
    
    Returns
    -------
    dict with arrays of real, imag, magnitude, angle, frequency
    """
    evals = np.asarray(evals).flatten()
    
    real = np.real(evals)
    imag = np.imag(evals)
    magnitude = np.abs(evals)
    angle = np.angle(evals)
    frequency = np.abs(angle) / (2 * np.pi) * fs
    
    return {
        'real': real,
        'imag': imag,
        'magnitude': magnitude,
        'angle': angle,
        'frequency': frequency
    }


print("✅ Helper functions loaded")

---
# SECTION 3 — Aggregate All Eigenvalues
> Load all eigenvalues into a single DataFrame

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# AGGREGATE EIGENVALUES — OPTIMIZED FOR LARGE DATASETS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("AGGREGATING EIGENVALUES (OPTIMIZED)")
print("="*70 + "\n")

# Configuration
SAMPLE_RATE = 1  # Take 1 every N eigenvalues to reduce data size
BATCH_SIZE = 1000  # Process files in batches

print(f"⚙️  Settings:")
print(f"   Sample rate: 1/{SAMPLE_RATE} eigenvalues (to reduce memory)")
print(f"   Batch size: {BATCH_SIZE} files")
print(f"   Expected final size: ~{len(df_inventory) * 132000 // SAMPLE_RATE:,} rows\n")

# Process in batches
t0 = time.time()
batch_files = []
all_batches = []

for idx, row in df_inventory.iterrows():
    try:
        # Load data
        result = load_npz_data(row['filepath'])
        evals = result['evals_flat']
        meta = result['metadata']
        
        # SAMPLE eigenvalues (take every Nth to reduce size)
        evals_sampled = evals[::SAMPLE_RATE]
        
        # Compute features
        feats = eigenvalue_features(evals_sampled, fs=meta['sfreq'])
        
        # Create batch dataframe for this file
        df_file = pd.DataFrame({
            'representation': row['representation'],
            'grupo': meta['grupo'],
            'cond': meta['cond'],
            'win_sec': meta['win_sec'],
            'subj_id': meta['subj_id'],
            'real': feats['real'],
            'imag': feats['imag'],
            'magnitude': feats['magnitude'],
            'frequency': feats['frequency'],
        })
        
        batch_files.append(df_file)
        
        # Save batch when it reaches BATCH_SIZE
        if len(batch_files) >= BATCH_SIZE:
            batch_df = pd.concat(batch_files, ignore_index=True)
            all_batches.append(batch_df)
            batch_files = []  # Clear for next batch
            print(f"   Processed {idx+1}/{len(df_inventory)} files... "
                  f"{len(all_batches)} batches ready ({tsec(t0)})")
            
    except Exception as e:
        print(f"⚠️  Error: {row['filename']}: {e}")

# Process remaining files
if batch_files:
    batch_df = pd.concat(batch_files, ignore_index=True)
    all_batches.append(batch_df)

# Concatenate all batches
print(f"\n   Concatenating {len(all_batches)} batches...")
df_agg = pd.concat(all_batches, ignore_index=True)

print(f"\n{'─'*70}")
print(f"✅ Aggregated {len(df_agg):,} eigenvalues in {tsec(t0)}")

if len(df_agg) > 0:
    print(f"\nBreakdown:")
    print(f"   Representations: {list(df_agg['representation'].unique())}")
    print(f"   Groups: {list(df_agg['grupo'].unique())}")
    print(f"   Conditions: {list(df_agg['cond'].unique())}")
    print(f"   Window sizes: {sorted(df_agg['win_sec'].unique())}")
    
    # Memory usage
    mem_mb = df_agg.memory_usage(deep=True).sum() / 1024**2
    print(f"   Memory usage: {mem_mb:.1f} MB")
    
    # Save
    agg_path = RUTA_FIGS / "_aggregated_eigenvalues.csv"
    print(f"\n   Saving to CSV (this may take a moment)...")
    df_agg.to_csv(agg_path, index=False)
    print(f"   💾 Saved: {agg_path}")
    print(f"   File size: {agg_path.stat().st_size / 1024**2:.1f} MB")
else:
    print("\n⚠️  No eigenvalues aggregated!")

---
# SECTION 4 — Group Comparison Figures
> Generate comparison figures for Gifted vs Control

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# GROUP COMPARISON — COMPLEX PLANE (OVERLAID)
# ══════════════════════════════════════════════════════════════════════════════

if len(df_agg) == 0:
    print("⚠️  No data to plot")
else:
    print("\n" + "="*70)
    print("GROUP COMPARISON — COMPLEX PLANE (OVERLAID)")
    print("="*70 + "\n")
    
    t0 = time.time()
    n_figs = 0
    
    combos = df_agg.groupby(['representation', 'cond', 'win_sec']).size().reset_index(name='count')
    
    for _, combo in combos.iterrows():
        rep = combo['representation']
        cond = combo['cond']
        win = combo['win_sec']
        
        # Filter data
        mask = (df_agg['representation'] == rep) & \
               (df_agg['cond'] == cond) & \
               (df_agg['win_sec'] == win)
        df_sub = df_agg[mask]
        
        if len(df_sub) < 100:
            continue
        
        # Split by group
        df_g = df_sub[df_sub['grupo'] == 'Gifted']
        df_c = df_sub[df_sub['grupo'] == 'Control']
        
        if len(df_g) < 50 or len(df_c) < 50:
            continue
        
        # Create SINGLE figure with BOTH groups overlaid
        fig, ax = plt.subplots(1, 1, figsize=(8, 8))
        
        # Common limits
        all_real = np.concatenate([df_g['real'], df_c['real']])
        all_imag = np.concatenate([df_g['imag'], df_c['imag']])
        max_lim = max(np.max(np.abs(all_real)), np.max(np.abs(all_imag)), 1.2)
        
        # Plot BOTH groups on SAME axes
        # Control first (blue, behind)
        ax.scatter(df_c['real'], df_c['imag'],
                  c=C_CTRL, alpha=0.3, s=8, edgecolors='none', 
                  label=f'Control (n={len(df_c):,})', rasterized=True)
        
        # Gifted second (red, on top)
        ax.scatter(df_g['real'], df_g['imag'],
                  c=C_GIFTED, alpha=0.3, s=8, edgecolors='none',
                  label=f'Gifted (n={len(df_g):,})', rasterized=True)
        
        # Unit circle
        circle = Circle((0, 0), 1, fill=False, edgecolor='black',
                       linewidth=2, linestyle='--', alpha=0.8)
        ax.add_patch(circle)
        
        # Axes
        ax.axhline(0, color='gray', linewidth=0.5, alpha=0.5)
        ax.axvline(0, color='gray', linewidth=0.5, alpha=0.5)
        ax.set_xlabel('Re(λ)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Im(λ)', fontsize=12, fontweight='bold')
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlim(-max_lim, max_lim)
        ax.set_ylim(-max_lim, max_lim)
        ax.grid(True, alpha=0.3)
        
        # Legend
        ax.legend(loc='upper left', fontsize=11, framealpha=0.9)
        
        # Title
        ax.set_title(f"{rep} | {cond.upper()} | {win}s",
                    fontsize=12, fontweight='bold', pad=15)
        
        plt.tight_layout()
        
        # Save
        folder = RUTA_FIGS / "group_comparison" / "complex_plane_overlaid" / rep / cond
        save_fig(fig, folder / f"complex_{win}s.png")
        plt.close(fig)
        n_figs += 1
    
    print(f"\n✅ Created {n_figs} overlaid complex plane figures in {tsec(t0)}")

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# GROUP COMPARISON — MAGNITUDE VS FREQUENCY (OVERLAID - SCATTER ONLY)
# ══════════════════════════════════════════════════════════════════════════════

if len(df_agg) == 0:
    print("⚠️  No data to plot")
else:
    print("\n" + "="*70)
    print("GROUP COMPARISON — MAGNITUDE VS FREQUENCY (OVERLAID)")
    print("="*70 + "\n")
    
    t0 = time.time()
    n_figs = 0
    
    combos = df_agg.groupby(['representation', 'cond', 'win_sec']).size().reset_index(name='count')
    
    for _, combo in combos.iterrows():
        rep = combo['representation']
        cond = combo['cond']
        win = combo['win_sec']
        
        # USE ALL EIGENVALUES
        mask = (df_agg['representation'] == rep) & \
               (df_agg['cond'] == cond) & \
               (df_agg['win_sec'] == win)
        df_sub = df_agg[mask]
        
        if len(df_sub) < 100:
            continue
        
        df_g = df_sub[df_sub['grupo'] == 'Gifted']
        df_c = df_sub[df_sub['grupo'] == 'Control']
        
        if len(df_g) < 50 or len(df_c) < 50:
            continue
        
        # Create SINGLE figure with BOTH groups
        fig, ax = plt.subplots(1, 1, figsize=(9, 7))
        
        # Limits
        max_mag = 1.2
        max_freq = 50
        
        # Filter for display
        df_c_disp = df_c[(df_c['magnitude'] <= max_mag) & (df_c['frequency'] <= max_freq)]
        df_g_disp = df_g[(df_g['magnitude'] <= max_mag) & (df_g['frequency'] <= max_freq)]
        
        print(f"   Control: {len(df_c_disp):,} / {len(df_c):,} points")
        print(f"   Gifted: {len(df_g_disp):,} / {len(df_g):,} points")
        
        # Plot with SCATTER ONLY (no hexbin!)
        # Control first (blue, behind)
        ax.scatter(df_c_disp['magnitude'], df_c_disp['frequency'],
                  c=C_CTRL, alpha=0.2, s=8, edgecolors='none',
                  label=f'Control (n={len(df_c_disp):,})', rasterized=True)
        
        # Gifted second (red, on top)
        ax.scatter(df_g_disp['magnitude'], df_g_disp['frequency'],
                  c=C_GIFTED, alpha=0.2, s=8, edgecolors='none',
                  label=f'Gifted (n={len(df_g_disp):,})', rasterized=True)
        
        # Unit magnitude line
        ax.axvline(1, color='black', linewidth=2, linestyle='--',
                  alpha=0.8, label='|λ| = 1 (stability)', zorder=5)
        
        ax.set_xlabel('Magnitude |λ|', fontsize=12, fontweight='bold')
        ax.set_ylabel('Frequency (Hz)', fontsize=12, fontweight='bold')
        ax.set_xlim(0, max_mag)
        ax.set_ylim(0, max_freq)
        ax.grid(True, alpha=0.3, linewidth=0.5)
        ax.legend(fontsize=11, loc='upper right', framealpha=0.9)
        ax.set_title(f"{rep} | {cond.upper()} | {win}s",
                    fontsize=12, fontweight='bold', pad=15)
        
        plt.tight_layout()
        
        # Save
        folder = RUTA_FIGS / "group_comparison" / "magnitude_frequency_overlaid" / rep / cond
        save_fig(fig, folder / f"magfreq_{win}s.png")
        plt.close(fig)
        n_figs += 1
    
    print(f"\n✅ Created {n_figs} overlaid magnitude-frequency figures in {tsec(t0)}")

---
# SECTION 5 — Priority Figures (PVT 8s & 4s)


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# PRIORITY  FIGURES
# ══════════════════════════════════════════════════════════════════════════════

if len(df_agg) == 0:
    print("⚠️  No data to plot")
else:
    print("\n" + "="*70)
    print("PRIORITY  FIGURES (PVT 8s & 4s)")
    print("="*70 + "\n")
    
    priority_folder = RUTA_FIGS / "paper_priority"
    t0 = time.time()
    
    for win in PRIORITY_WINDOWS:
        for rep in ['all_channels', 'no_occipital']:
            
            # Filter for PVT
            mask = (df_agg['cond'] == 'pvt') & \
                   (df_agg['win_sec'] == win) & \
                   (df_agg['representation'] == rep)
            df_sub = df_agg[mask]
            
            if len(df_sub) < 100:
                print(f"⚠️  Insufficient data for {rep}, PVT, {win}s")
                continue
            
            df_g = df_sub[df_sub['grupo'] == 'Gifted']
            df_c = df_sub[df_sub['grupo'] == 'Control']
            
            if len(df_g) < 50 or len(df_c) < 50:
                continue
            
            # Create combined figure: Complex + MagFreq
            fig = plt.figure(figsize=(16, 7))
            gs = gridspec.GridSpec(1, 4, figure=fig, wspace=0.3)
            
            # Common limits
            all_real = np.concatenate([df_g['real'], df_c['real']])
            all_imag = np.concatenate([df_g['imag'], df_c['imag']])
            max_lim = max(np.max(np.abs(all_real)), np.max(np.abs(all_imag)), 1.2)
            
            # Upper half-plane for mag-freq
            df_sub_upper = df_sub[df_sub['imag'] > TAU_IMAG]
            df_g_upper = df_sub_upper[df_sub_upper['grupo'] == 'Gifted']
            df_c_upper = df_sub_upper[df_sub_upper['grupo'] == 'Control']
            
            max_mag = 1.5
            max_freq = 120
            
            # ── Complex Plane Panels ──
            for idx, (df_group, grupo, color) in enumerate([
                (df_c, 'Control', C_CTRL),
                (df_g, 'Gifted', C_GIFTED)
            ]):
                ax = fig.add_subplot(gs[0, idx])
                ax.scatter(df_group['real'], df_group['imag'],
                          c=color, alpha=0.15, s=8, edgecolors='none', rasterized=True)
                circle = Circle((0, 0), 1, fill=False, edgecolor='black',
                               linewidth=1.5, linestyle='--', alpha=0.7)
                ax.add_patch(circle)
                ax.axhline(0, color='gray', linewidth=0.5, alpha=0.5)
                ax.axvline(0, color='gray', linewidth=0.5, alpha=0.5)
                ax.set_xlabel('Re(λ)', fontsize=10)
                ax.set_ylabel('Im(λ)', fontsize=10)
                ax.set_aspect('equal', adjustable='box')
                ax.set_xlim(-max_lim, max_lim)
                ax.set_ylim(-max_lim, max_lim)
                ax.grid(True, alpha=0.3)
                ax.set_title(f"{grupo}\n(n={len(df_group):,})",
                            fontsize=11, fontweight='bold', color=color)
            
            # ── Magnitude-Frequency Panels ──
            for idx, (df_group, grupo, color) in enumerate([
                (df_c_upper, 'Control', C_CTRL),
                (df_g_upper, 'Gifted', C_GIFTED)
            ]):
                ax = fig.add_subplot(gs[0, idx + 2])
                
                if len(df_group) > 1000:
                    ax.hexbin(df_group['magnitude'], df_group['frequency'],
                             gridsize=35, cmap='Greys', alpha=0.7, mincnt=1,
                             extent=[0, max_mag, 0, max_freq])
                else:
                    ax.scatter(df_group['magnitude'], df_group['frequency'],
                              c=color, alpha=0.3, s=12, edgecolors='none', rasterized=True)
                
                ax.axvline(1, color='black', linewidth=1.5, linestyle='--', alpha=0.7)
                ax.set_xlabel('|λ|', fontsize=10)
                ax.set_ylabel('Frequency (Hz)', fontsize=10)
                ax.set_xlim(0, max_mag)
                ax.set_ylim(0, max_freq)
                ax.grid(True, alpha=0.3)
                ax.set_title(f"{grupo}\n(n={len(df_group):,})",
                            fontsize=11, fontweight='bold', color=color)
            
            fig.suptitle(f"Eigenvalue Comparison — PVT Task ({win}s windows)\n" +
                        f"Representation: {rep} | Left: Complex Plane | Right: Magnitude-Frequency",
                        fontsize=13, fontweight='bold', y=0.98)
            
            plt.tight_layout()
            save_fig(fig, priority_folder / f"PVT_{rep}_{int(win)}s_combined.png")
            plt.close(fig)
    
    print(f"\n✅ Priority figures completed in {tsec(t0)}")

---
# SECTION 6 — Eigenvector Analysis: Spatial Participation
> Analyze which brain regions participate in each eigenmode

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# CHANNEL POSITIONS FOR TOPOPLOTS
# ══════════════════════════════════════════════════════════════════════════════

# Standard 10-20 positions (normalized coordinates)
CHANNEL_POSITIONS = {
    'Fp1': (-0.30, 0.90), 'Fp2': (0.30, 0.90), 'Fpz': (0.00, 0.95),
    'AF3': (-0.25, 0.78), 'AF4': (0.25, 0.78),
    'F7': (-0.70, 0.60), 'F3': (-0.40, 0.65), 'Fz': (0.00, 0.72),
    'F4': (0.40, 0.65), 'F8': (0.70, 0.60),
    'FC5': (-0.60, 0.42), 'FC1': (-0.22, 0.50), 'FC2': (0.22, 0.50), 'FC6': (0.60, 0.42),
    'T7': (-0.88, 0.00), 'C3': (-0.45, 0.00), 'Cz': (0.00, 0.00),
    'C4': (0.45, 0.00), 'T8': (0.88, 0.00),
    'CP5': (-0.60, -0.42), 'CP1': (-0.22, -0.50), 'CP2': (0.22, -0.50), 'CP6': (0.60, -0.42),
    'P7': (-0.70, -0.60), 'P3': (-0.40, -0.65), 'Pz': (0.00, -0.72),
    'P4': (0.40, -0.65), 'P8': (0.70, -0.60),
    'O1': (-0.30, -0.90), 'Oz': (0.00, -0.95), 'O2': (0.30, -0.90),
    'POz': (0.00, -0.82),
}

# Brain regions
REGIONS = {
    'Frontal': ['Fp1', 'Fp2', 'Fpz', 'AF3', 'AF4', 'F7', 'F3', 'Fz', 'F4', 'F8'],
    'Central': ['FC5', 'FC1', 'FC2', 'FC6', 'C3', 'Cz', 'C4', 'CP1', 'CP2', 'CP5', 'CP6'],
    'Temporal': ['T7', 'T8'],
    'Parietal': ['P7', 'P3', 'Pz', 'P4', 'P8'],
    'Occipital': ['O1', 'Oz', 'O2', 'POz'],
}

print("✅ Channel positions loaded")
print(f"   Total channels: {len(CHANNEL_POSITIONS)}")
print(f"   Regions: {list(REGIONS.keys())}")

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPUTE SPATIAL PARTICIPATION FROM EIGENVECTORS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("COMPUTING SPATIAL PARTICIPATION")
print("="*70 + "\n")

participation_data = []
t0 = time.time()

for idx, row in df_inventory.iterrows():
    try:
        data_npz = np.load(row['filepath'], allow_pickle=True)
        
        evecs = data_npz['evecs']  # (n_segments, n_channels, n_channels)
        ch_names = data_npz['ch_names']
        grupo = standardize_group(data_npz.get('group', 'unknown'))
        cond = str(to_scalar(data_npz.get('cond', 'unknown'))).lower()
        win_sec = to_scalar(data_npz.get('win_sec', None))
        
        # Compute participation: |eigenvector|^2 normalized
        # For each mode, sum across all segments to get average participation
        participation = np.abs(evecs) ** 2  # (n_seg, n_ch, n_modes)
        
        # Average across segments
        avg_participation = participation.mean(axis=0)  # (n_ch, n_modes)
        
        # Average across all modes to get per-channel participation
        channel_participation = avg_participation.mean(axis=1)  # (n_ch,)
        
        # Normalize
        channel_participation = channel_participation / channel_participation.sum()
        
        # Store
        participation_data.append({
            'representation': row['representation'],
            'grupo': grupo,
            'cond': cond,
            'win_sec': win_sec,
            'subj_id': row['subj_id'],
            'ch_names': ch_names,
            'participation': channel_participation,
            'participation_by_mode': avg_participation,  # For detailed analysis
        })
        
        data_npz.close()
        
        if (idx + 1) % 20 == 0:
            print(f"   Processed {idx+1}/{len(df_inventory)} files...")
            
    except Exception as e:
        print(f"⚠️  Error: {row['filename']}: {e}")

print(f"\n✅ Participation computed for {len(participation_data)} files in {tsec(t0)}")

---
# SECTION 7 — Topoplots: Average Spatial Participation
> Visualize which brain regions are most active per group

In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# TOPOPLOTS: GIFTED VS CONTROL
# ══════════════════════════════════════════════════════════════════════════════

from scipy.interpolate import griddata

def plot_topomap(ax, ch_names, values, title, cmap='YlOrRd', vmin=None, vmax=None):
    """
    Plot a topographic map of channel values.
    """
    # Get positions
    positions = []
    vals = []
    for ch, val in zip(ch_names, values):
        if ch in CHANNEL_POSITIONS:
            positions.append(CHANNEL_POSITIONS[ch])
            vals.append(val)
    
    positions = np.array(positions)
    vals = np.array(vals)
    
    # Create interpolation grid
    grid_x, grid_y = np.mgrid[-1:1:100j, -1:1:100j]
    
    # Interpolate
    grid_z = griddata(positions, vals, (grid_x, grid_y), method='cubic')
    
    # Mask outside head
    mask = (grid_x**2 + grid_y**2) > 1.0
    grid_z[mask] = np.nan
    
    # Plot
    im = ax.contourf(grid_x, grid_y, grid_z, levels=20, cmap=cmap, 
                     vmin=vmin, vmax=vmax, extend='both')
    
    # Head outline
    head = Circle((0, 0), 1.0, fill=False, edgecolor='black', linewidth=2)
    ax.add_patch(head)
    
    # Nose
    nose_x = [0.15, 0, -0.15]
    nose_y = [1.05, 1.15, 1.05]
    ax.plot(nose_x, nose_y, 'k-', linewidth=2)
    
    # Electrodes
    ax.scatter(positions[:, 0], positions[:, 1], 
              c='white', s=15, edgecolors='black', linewidths=0.5, zorder=10)
    
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.3)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=11, fontweight='bold', pad=10)
    
    return im


# Generate topoplots for PVT 8s
if len(participation_data) > 0:
    print("\n" + "="*70)
    print("GENERATING TOPOPLOTS")
    print("="*70 + "\n")
    
    topoplot_folder = RUTA_FIGS / "topoplots"
    
    for rep in ['all_channels', 'no_occipital']:
        for win in [8.0, 4.0]:
            # Filter data
            subset = [p for p in participation_data 
                     if p['representation'] == rep and 
                        p['cond'] == 'pvt' and 
                        p['win_sec'] == win]
            
            if len(subset) < 4:
                continue
            
            # Separate by group
            gifted_data = [p for p in subset if p['grupo'] == 'Gifted']
            control_data = [p for p in subset if p['grupo'] == 'Control']
            
            if len(gifted_data) < 2 or len(control_data) < 2:
                continue
            
            # Average participation per group
            ch_names = gifted_data[0]['ch_names']
            
            gifted_avg = np.mean([p['participation'] for p in gifted_data], axis=0)
            control_avg = np.mean([p['participation'] for p in control_data], axis=0)
            
            # Create figure
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            
            # Common colorbar limits
            vmin = min(gifted_avg.min(), control_avg.min())
            vmax = max(gifted_avg.max(), control_avg.max())
            
            # Control
            im1 = plot_topomap(axes[0], ch_names, control_avg, 
                              f"Control\n(n={len(control_data)})",
                              vmin=vmin, vmax=vmax)
            
            # Gifted
            im2 = plot_topomap(axes[1], ch_names, gifted_avg, 
                              f"Gifted\n(n={len(gifted_data)})",
                              vmin=vmin, vmax=vmax)
            
            # Difference (Gifted - Control)
            diff = gifted_avg - control_avg
            im3 = plot_topomap(axes[2], ch_names, diff, 
                              "Gifted - Control",
                              cmap='RdBu_r', vmin=-diff.max(), vmax=diff.max())
            
            # Colorbars
            fig.colorbar(im2, ax=axes[:2], fraction=0.02, pad=0.04, 
                        label='Participation')
            fig.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04, 
                        label='Difference')
            
            fig.suptitle(f"Spatial Participation — {rep} | PVT | {win}s",
                        fontsize=13, fontweight='bold', y=0.98)
            
            plt.tight_layout()
            save_fig(fig, topoplot_folder / rep / f"topoplot_PVT_{win}s.png")
            plt.close(fig)
    
    print("\n✅ Topoplots generated")
else:
    print("⚠️  No participation data available")

---
# SECTION 8 — Connectivity Heatmaps
> Analyze communication between brain regions

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# CONNECTIVITY ANALYSIS FROM EIGENVECTORS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("COMPUTING REGIONAL CONNECTIVITY")
print("="*70 + "\n")

def compute_regional_connectivity(evecs, ch_names, regions_dict):
    """
    Compute connectivity between brain regions from eigenvectors.
    
    Returns
    -------
    connectivity_matrix : (n_regions, n_regions) array
        Averaged connectivity strength between regions
    """
    # Map channels to regions
    ch_to_region = {}
    for region, channels in regions_dict.items():
        for ch in channels:
            ch_to_region[ch] = region
    
    # Get region indices for each channel
    region_names = list(regions_dict.keys())
    n_regions = len(region_names)
    
    connectivity = np.zeros((n_regions, n_regions))
    counts = np.zeros((n_regions, n_regions))
    
    # Average across all segments and modes
    evecs_avg = np.abs(evecs).mean(axis=0)  # Average across segments: (n_ch, n_modes)
    
    # For each mode, compute co-activation between regions
    for mode_idx in range(evecs_avg.shape[1]):
        mode_vec = evecs_avg[:, mode_idx]
        
        # Compute pairwise products (co-activation)
        for i, ch_i in enumerate(ch_names):
            if ch_i not in ch_to_region:
                continue
            region_i = region_names.index(ch_to_region[ch_i])
            
            for j, ch_j in enumerate(ch_names):
                if ch_j not in ch_to_region:
                    continue
                region_j = region_names.index(ch_to_region[ch_j])
                
                # Co-activation strength
                connectivity[region_i, region_j] += mode_vec[i] * mode_vec[j]
                counts[region_i, region_j] += 1
    
    # Normalize
    connectivity = np.divide(connectivity, counts, 
                            where=counts > 0, out=np.zeros_like(connectivity))
    
    return connectivity, region_names


# Compute connectivity for each file
connectivity_data = []
t0 = time.time()

for idx, row in df_inventory.iterrows():
    try:
        data_npz = np.load(row['filepath'], allow_pickle=True)
        
        evecs = data_npz['evecs']
        ch_names = data_npz['ch_names']
        
        # Compute connectivity
        conn_matrix, region_names = compute_regional_connectivity(evecs, ch_names, REGIONS)
        
        connectivity_data.append({
            'representation': row['representation'],
            'grupo': row['grupo'],
            'cond': row['cond'],
            'win_sec': row['win_sec'],
            'subj_id': row['subj_id'],
            'connectivity': conn_matrix,
            'region_names': region_names,
        })
        
        data_npz.close()
        
        if (idx + 1) % 20 == 0:
            print(f"   Processed {idx+1}/{len(df_inventory)} files...")
            
    except Exception as e:
        print(f"⚠️  Error: {row['filename']}: {e}")

print(f"\n✅ Connectivity computed for {len(connectivity_data)} files in {tsec(t0)}")

In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# CONNECTIVITY HEATMAPS: GIFTED VS CONTROL
# ══════════════════════════════════════════════════════════════════════════════

if len(connectivity_data) > 0:
    print("\n" + "="*70)
    print("GENERATING CONNECTIVITY HEATMAPS")
    print("="*70 + "\n")
    
    heatmap_folder = RUTA_FIGS / "connectivity_heatmaps"
    
    for rep in ['all_channels', 'no_occipital']:
        for cond in ['basal', 'pvt']:
            for win in [8.0, 4.0]:
                # Filter
                subset = [c for c in connectivity_data 
                         if c['representation'] == rep and 
                            c['cond'] == cond and 
                            c['win_sec'] == win]
                
                if len(subset) < 4:
                    continue
                
                # Separate by group
                gifted = [c for c in subset if c['grupo'] == 'Gifted']
                control = [c for c in subset if c['grupo'] == 'Control']
                
                if len(gifted) < 2 or len(control) < 2:
                    continue
                
                # Average connectivity
                conn_gifted = np.mean([c['connectivity'] for c in gifted], axis=0)
                conn_control = np.mean([c['connectivity'] for c in control], axis=0)
                conn_diff = conn_gifted - conn_control
                
                region_names = gifted[0]['region_names']
                
                # Create figure
                fig, axes = plt.subplots(1, 3, figsize=(18, 5))
                
                # Common colorbar limits
                vmin = min(conn_gifted.min(), conn_control.min())
                vmax = max(conn_gifted.max(), conn_control.max())
                
                # Control
                im1 = axes[0].imshow(conn_control, cmap='YlOrRd', 
                                    vmin=vmin, vmax=vmax, aspect='auto')
                axes[0].set_title(f"Control (n={len(control)})", 
                                 fontsize=11, fontweight='bold')
                axes[0].set_xticks(range(len(region_names)))
                axes[0].set_yticks(range(len(region_names)))
                axes[0].set_xticklabels(region_names, rotation=45, ha='right')
                axes[0].set_yticklabels(region_names)
                plt.colorbar(im1, ax=axes[0], fraction=0.046)
                
                # Gifted
                im2 = axes[1].imshow(conn_gifted, cmap='YlOrRd', 
                                    vmin=vmin, vmax=vmax, aspect='auto')
                axes[1].set_title(f"Gifted (n={len(gifted)})", 
                                 fontsize=11, fontweight='bold')
                axes[1].set_xticks(range(len(region_names)))
                axes[1].set_yticks(range(len(region_names)))
                axes[1].set_xticklabels(region_names, rotation=45, ha='right')
                axes[1].set_yticklabels(region_names)
                plt.colorbar(im2, ax=axes[1], fraction=0.046)
                
                # Difference
                diff_max = np.abs(conn_diff).max()
                im3 = axes[2].imshow(conn_diff, cmap='RdBu_r', 
                                    vmin=-diff_max, vmax=diff_max, aspect='auto')
                axes[2].set_title("Gifted - Control", 
                                 fontsize=11, fontweight='bold')
                axes[2].set_xticks(range(len(region_names)))
                axes[2].set_yticks(range(len(region_names)))
                axes[2].set_xticklabels(region_names, rotation=45, ha='right')
                axes[2].set_yticklabels(region_names)
                plt.colorbar(im3, ax=axes[2], fraction=0.046)
                
                fig.suptitle(f"Regional Connectivity — {rep} | {cond.upper()} | {win}s",
                            fontsize=13, fontweight='bold', y=0.98)
                
                plt.tight_layout()
                save_fig(fig, heatmap_folder / rep / cond / f"connectivity_{win}s.png")
                plt.close(fig)
    
    print("\n✅ Connectivity heatmaps generated")
else:
    print("⚠️  No connectivity data available")

---
# FINAL SUMMARY
> Inventory of generated figures

In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70 + "\n")

# Count all PNG files
all_pngs = list(RUTA_FIGS.rglob("*.png"))
all_csvs = list(RUTA_FIGS.rglob("*.csv"))

print(f"📊 Generated {len(all_pngs)} figures")
print(f"📄 Created {len(all_csvs)} data files\n")

# Group by folder
folders = {}
for png in all_pngs:
    rel_folder = png.parent.relative_to(RUTA_FIGS)
    if str(rel_folder) not in folders:
        folders[str(rel_folder)] = 0
    folders[str(rel_folder)] += 1

print("By folder:")
for folder in sorted(folders.keys()):
    print(f"   {folder:50s}: {folders[folder]:3d} files")

# Priority files check
print("\n" + "─"*70)

print("─"*70)

priority_files = [
    "paper_priority/PVT_no_occipital_8s_combined.png",
    "paper_priority/PVT_no_occipital_4s_combined.png",
    "paper_priority/PVT_all_channels_8s_combined.png",
    "paper_priority/PVT_all_channels_4s_combined.png",
]

for pf in priority_files:
    full_path = RUTA_FIGS / pf
    if full_path.exists():
        size_kb = full_path.stat().st_size / 1024
        print(f"   ✓ {pf:60s} {size_kb:7.0f} KB")
    else:
        print(f"   ✗ {pf} (not found)")

print("\n" + "="*70)
print("✅ NOTEBOOK EXECUTION COMPLETED")
print("="*70)
print(f"\nAll outputs saved to: {RUTA_FIGS}\n")